# DiscoGeM 1.0 — Data Inspection

First deliverable per the project spec: inspect the raw DiscoGeM 1.0 CSVs and confirm the schema assumptions used by Stage 1 (Prep) of the pipeline.

Run this notebook before any LLM calls. No API keys needed.

**What we verify here:**
1. Column names + dtypes of both CSVs
2. Per-item argument pairs and aggregated labels (wide format)
3. Per-annotator Level-2 sense distribution (`lev2_conn2` from full dataset)
4. Genre distribution and the three-genre mapping
5. NA / multi-valued handling for `lev2_conn2`
6. Stratified sampling dry run with seed=42

In [ ]:
from collections import Counter
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_colwidth', 80)
sns.set_style('whitegrid')

PROJECT_ROOT = Path.cwd().parent
WIDE_CSV = PROJECT_ROOT / 'DiscoGeM 1.0_items' / 'DiscoGeM1.0.wide.csv'
FULL_CSV = PROJECT_ROOT / 'DiscoGeM 1.0_labels' / 'DiscoGeMcorpus_fulldataset.csv'

print('Wide CSV exists:', WIDE_CSV.exists())
print('Full CSV exists:', FULL_CSV.exists())

## 1. Load wide format (one row per item)

In [ ]:
wide = pd.read_csv(WIDE_CSV, low_memory=False)
print(f'shape: {wide.shape}')
print()
print('columns:')
for c in wide.columns:
    print(f'  {c:35s} {wide[c].dtype}')
wide.head(5)

## 2. Load full dataset (one row per worker x item)

In [ ]:
full = pd.read_csv(FULL_CSV, low_memory=False)
print(f'shape: {full.shape}')
print()
for c in full.columns:
    print(f'  {c:25s} {full[c].dtype}')
full[['itemid', 'workerid', 'firstinput', 'secondinput', 'lev2_conn2', 'lev3_conn2']].head(5)

## 3. Genre distribution

The pipeline operates on three genres: Europarl, Lit (novel), Wiki. PDTB items are excluded.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
wide['genre'].value_counts().plot(kind='bar', ax=ax[0], title='Items (wide format)')
full['genre'].value_counts().plot(kind='bar', ax=ax[1], title='Annotations (full dataset)')
plt.tight_layout()
plt.show()
wide['genre'].value_counts()

## 4. `maxagree` distribution per genre

Stratification bins: high (>= 0.5) and low (< 0.5).

In [ ]:
GENRE_MAP = {'novel': 'Lit', 'europarl': 'Europarl', 'wikipedia': 'Wiki'}
wide_three = wide[wide['genre'].isin(GENRE_MAP)].copy()
wide_three['genre_mapped'] = wide_three['genre'].map(GENRE_MAP)

fig, ax = plt.subplots(figsize=(8, 4))
for genre in ['Europarl', 'Lit', 'Wiki']:
    sub = wide_three[wide_three['genre_mapped'] == genre]['maxagree']
    ax.hist(sub, bins=20, alpha=0.55, label=genre)
ax.axvline(0.5, color='red', linestyle='--', label='high/low cutoff (0.5)')
ax.set_xlabel('maxagree (fraction)')
ax.set_ylabel('item count')
ax.set_title('Per-genre maxagree distribution')
ax.legend()
plt.tight_layout()
plt.show()

wide_three.groupby('genre_mapped')['maxagree'].describe()

## 5. `lev2_conn2` value counts and NA / multi-valued handling

In [ ]:
lev2 = full['lev2_conn2'].astype(str)
n_total = len(lev2)
n_na = (lev2.str.upper() == 'NA').sum() + lev2.isna().sum()
n_multi = lev2.str.contains(',', na=False).sum()
print(f'total annotations: {n_total}')
print(f'  NA: {n_na} ({n_na / n_total:.1%})')
print(f'  multi-valued (comma): {n_multi} ({n_multi / n_total:.1%})')
print()
single_lev2 = lev2[~lev2.str.contains(',', na=False) & (lev2.str.upper() != 'NA') & lev2.notna()]
print('unique single-valued Level-2 senses:')
print(sorted(single_lev2.unique()))
print()
print('top counts:')
print(single_lev2.value_counts().head(20))

## 6. Level-2 sense frequency bar chart

In [ ]:
counts = single_lev2.value_counts()
plt.figure(figsize=(9, 5))
counts.plot(kind='bar')
plt.ylabel('annotation count')
plt.title('PDTB Level-2 sense frequency (Step-2 selections, all genres)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 7. Per-item crowd distributions (3 examples, one per genre)

Group `lev2_conn2` by `itemid` (excluding NA and multi-valued) to get the per-annotator senses. Compute the normalized distribution — this is the **crowd ground truth** the pipeline compares against.

In [ ]:
full_clean = full[
    full['lev2_conn2'].notna()
    & (full['lev2_conn2'].astype(str).str.upper() != 'NA')
    & ~full['lev2_conn2'].astype(str).str.contains(',', na=False)
    & full['genre'].isin(GENRE_MAP)
].copy()

def crowd_distribution(senses):
    counts = Counter(senses)
    total = sum(counts.values())
    return {s: c / total for s, c in counts.items()}

groups = full_clean.groupby('itemid')['lev2_conn2'].apply(list).reset_index()
groups = groups.merge(wide_three[['itemid', 'genre_mapped']], on='itemid')

for genre in ['Europarl', 'Lit', 'Wiki']:
    example = groups[groups['genre_mapped'] == genre].iloc[0]
    dist = crowd_distribution(example['lev2_conn2'])
    print(f'=== {genre} :: {example["itemid"]} ===')
    print(f'  n annotations: {len(example["lev2_conn2"])}')
    print(f'  distribution: {dist}')
    print()

## 8. `arg1` vs `arg1_singlesentence` comparison

Per design decision: the pipeline uses the full-context `arg1`/`arg2` (with surrounding sentences). The single-sentence versions are still stored in the schema for reference.

In [ ]:
for i, row in wide_three.head(3).iterrows():
    print(f'=== {row["itemid"]} ===')
    print(f'arg1 (full):   {row["arg1"][:200]}...')
    print(f'arg1 (single): {row["arg1_singlesentence"][:200]}')
    print()

## 9. Stratified sampling dry run (seed=42, 50 items)

Verify that we can fill the 17 / 17 / 16 genre quotas with the half-high / half-low split.

In [ ]:
import random
import math

groups['crowd_dist'] = groups['lev2_conn2'].apply(crowd_distribution)
groups['crowd_agreement_score'] = groups['crowd_dist'].apply(lambda d: max(d.values()))
groups['stratification_bin'] = groups['crowd_agreement_score'].apply(
    lambda v: 'high' if v >= 0.5 else 'low'
)

print('Bin counts available:')
print(groups.groupby(['genre_mapped', 'stratification_bin']).size())
print()

GENRE_QUOTAS = {'Europarl': 17, 'Lit': 17, 'Wiki': 16}

def split_high_low(quota):
    high = math.ceil(quota / 2)
    return high, quota - high

rng = random.Random(42)
selected = []
for genre, quota in GENRE_QUOTAS.items():
    high_n, low_n = split_high_low(quota)
    pool = groups[groups['genre_mapped'] == genre]
    high_pool = pool[pool['stratification_bin'] == 'high'].sort_values('itemid')
    low_pool = pool[pool['stratification_bin'] == 'low'].sort_values('itemid')
    selected.extend(rng.sample(list(high_pool['itemid']), high_n))
    selected.extend(rng.sample(list(low_pool['itemid']), low_n))

selected_df = groups[groups['itemid'].isin(selected)]
print(f'\nSelected {len(selected_df)} items')
print(selected_df.groupby(['genre_mapped', 'stratification_bin']).size())

## 10. `observations` count distribution

Confirms most items have 10 annotators. A few have 11/17/19 — the crowd distribution normalizes by count so this is handled automatically.

In [ ]:
wide_three['observations'].value_counts().sort_index()